# Brain tumor segmentation: training on Kaggle

Thin wrapper around the [`btseg`](https://github.com/RoshanKarthik/brain-tumor-seg) package.
All logic lives in the repo; this notebook only clones it, finds the data and runs the scripts.

**Notebook settings (right-hand panel)**
- *Accelerator*: **GPU T4 x2** (training uses one GPU). Prefer T4 over P100: recent PyTorch builds may no longer support the P100.
- *Internet*: **on** (needed for `git clone` / `pip`). Requires a phone-verified Kaggle account.
- *Input*: add the dataset **BraTS 2021 Task 1** (`dschettler8845/brats-2021-task1`).
- *Secrets* (optional, only for the upload cell): `HF_TOKEN` = a Hugging Face **write** token. Never paste tokens into cells.

**How to run**
1. Interactive session, `RUN_SMOKE_TEST = True`, `RUN_FULL_TRAINING = False`: run all cells. Checks the data and measures time per epoch (~15 min).
2. Full training: set `RUN_SMOKE_TEST = False`, `RUN_FULL_TRAINING = True`, then **Save Version → Save & Run All (Commit)** so it keeps running in the background (max 12 h; training stops cleanly after `TIME_LIMIT_HOURS`).
3. To continue after a session ends: **Add Input → Your Work →** this notebook's previous version output, then Save & Run All again. The training cell finds the previous `last.pt` and resumes.

Outputs land in `/kaggle/working` (limited to ~20 GB): `runs/<run_name>/` with `last.pt`, `best.pt`, `train_log.csv`, and `splits.json` the first time.

In [ ]:
# ---- settings ----
REPO_URL = "https://github.com/RoshanKarthik/brain-tumor-seg.git"
BRANCH = "main"
CONFIG = "configs/segresnet.yaml"   # or configs/unet3d.yaml for the baseline

RUN_SMOKE_TEST = True       # 2 epochs on 20 cases: checks everything and times an epoch
RUN_FULL_TRAINING = False   # the real run (use Save & Run All)
TIME_LIMIT_HOURS = 11.0     # stop between epochs before Kaggle's 12 h limit

HF_REPO_ID = ""             # e.g. "<hf-username>/brats-segresnet"; empty = do not upload

WORK = "/kaggle/working"
REPO_DIR = "/tmp/brain-tumor-seg"   # code lives outside /kaggle/working (not part of the output)
SCRATCH = "/tmp/brats"              # extracted data, also outside /kaggle/working

In [ ]:
# ---- 1. clone the repo and install btseg (keeps Kaggle's preinstalled torch) ----
import os, sys
!rm -rf {REPO_DIR}
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="commit %h  %s"
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")  # the running kernel does not see the new editable install

import torch, monai
print("torch", torch.__version__, "| monai", monai.__version__, "| python", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (turn on the GPU accelerator)")

In [ ]:
# ---- 2. locate or extract the BraTS data ----
from btseg.data.prepare import prepare_data
from btseg.data.discover import discover_cases

DATA_ROOT = prepare_data("/kaggle/input", SCRATCH)
cases = discover_cases(DATA_ROOT, require_seg=True)
print(len(cases), "complete cases, e.g.", cases[0]["id"], "...", cases[-1]["id"])

In [ ]:
# ---- 3. patient-level split (seed 42, 80/10/10) ----
import json, shutil
from btseg.data.dataset import load_splits, select_split

SPLITS = f"{REPO_DIR}/data/splits.json"
if os.path.exists(SPLITS):
    print("Using the committed data/splits.json")
else:
    !python scripts/make_splits.py --data-root {DATA_ROOT} --out {SPLITS}
    shutil.copy(SPLITS, f"{WORK}/splits.json")
    print("NEW SPLIT: download /kaggle/working/splits.json and commit it to the repo as data/splits.json")

splits = load_splits(SPLITS)
for name in ("train", "val", "test"):
    print(name, len(select_split(cases, splits, name)))  # fails if the split does not match the data

In [ ]:
# ---- 4. smoke test: 2 epochs on 20 train / 5 val cases, then estimate a full epoch ----
import csv
if RUN_SMOKE_TEST:
    SMOKE_DIR = f"{WORK}/smoke"
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    !python -m btseg.train --config {CONFIG} --data-root {DATA_ROOT} --splits {SPLITS} --out {SMOKE_DIR} --epochs 2 --max-train-cases 20 --max-val-cases 5

    rows = list(csv.DictReader(open(f"{SMOKE_DIR}/train_log.csv")))
    e2_s = float(rows[1]["epoch_seconds"])  # epoch 2 = train + val; epoch 1 includes warm-up
    print(f"epoch 1: {float(rows[0]['epoch_seconds']):.0f}s (train, incl. warm-up), "
          f"epoch 2: {e2_s:.0f}s (train + 5 val cases)")
    print("Use these to pick train.epochs / val_interval; rough full-epoch estimate:",
          f"{float(rows[0]['epoch_seconds']) / 20 * len(splits['train']) / 60:.0f} min of training")

In [ ]:
# ---- 5. full training (resumable) ----
import glob, yaml
if RUN_FULL_TRAINING:
    run_name = yaml.safe_load(open(CONFIG))["run_name"]
    RUN_DIR = f"{WORK}/runs/{run_name}"
    if not os.path.exists(f"{RUN_DIR}/last.pt"):
        previous = sorted(glob.glob(f"/kaggle/input/**/runs/{run_name}/last.pt", recursive=True))
        if previous:
            print("Resuming from previous version output:", previous[-1])
            shutil.copytree(os.path.dirname(previous[-1]), RUN_DIR, dirs_exist_ok=True)
    RESUME = "--resume" if os.path.exists(f"{RUN_DIR}/last.pt") else ""
    !python -m btseg.train --config {CONFIG} --data-root {DATA_ROOT} --splits {SPLITS} --out {RUN_DIR} {RESUME} --time-limit-hours {TIME_LIMIT_HOURS}
    !tail -n 5 {RUN_DIR}/train_log.csv

In [ ]:
# ---- 6. optional: upload best.pt to the Hugging Face Hub (token from Kaggle Secrets) ----
if RUN_FULL_TRAINING and HF_REPO_ID and os.path.exists(f"{RUN_DIR}/best.pt"):
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi

    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)
    for name in ("best.pt", "config.json", "train_log.csv"):
        api.upload_file(path_or_fileobj=f"{RUN_DIR}/{name}", path_in_repo=f"{run_name}/{name}",
                        repo_id=HF_REPO_ID, repo_type="model")
    print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}/tree/main/{run_name}")